In [1]:
import cv2
import easyocr
from ultralytics import YOLO
import requests
import os

# === 1. Download model from Hugging Face if not already present ===
url = "https://huggingface.co/MKgoud/License-Plate-Recognizer/resolve/main/LP-detection.pt"
model_path = "LP-detection.pt"

if not os.path.exists(model_path):
    r = requests.get(url)
    with open(model_path, "wb") as f:
        f.write(r.content)

# === 2. Load YOLO model ===
model = YOLO(model_path)

# === 3. Initialize OCR ===
reader = easyocr.Reader(['en'])

# === 4. Output file ===
output_file = "detected_plates.txt"
open(output_file, "w").close()  # clear file at start

# === 5. Start webcam capture ===
cap = cv2.VideoCapture(0)  # change to 1/2 if multiple webcams

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy().astype(int)
        for (x_min, y_min, x_max, y_max) in boxes:
            # Draw bounding box
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

            # Crop plate
            plate_crop = frame[y_min:y_max, x_min:x_max]

            # OCR on plate
            ocr_result = reader.readtext(plate_crop)
            if ocr_result:
                plate_text = ocr_result[0][1].strip()
                cv2.putText(frame, plate_text, (x_min, y_min - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                print("Detected Plate Text:", plate_text)

                # Save to file
                with open(output_file, "a") as f:
                    f.write(plate_text + "\n")

    cv2.imshow("License Plate Detection", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print(f"All detected plates saved to {output_file}")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



0: 480x640 (no detections), 258.8ms
Speed: 6.2ms preprocess, 258.8ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 232.5ms
Speed: 6.1ms preprocess, 232.5ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 194.8ms
Speed: 4.5ms preprocess, 194.8ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 214.9ms
Speed: 3.6ms preprocess, 214.9ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 282.9ms
Speed: 3.0ms preprocess, 282.9ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 184.9ms
Speed: 4.8ms preprocess, 184.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 205.0ms
Speed: 3.1ms preprocess, 205.0ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 255.0ms
Speed: 3.9ms prepr